In [1]:
from typing import List, TypedDict
import time
from langchain_ollama import ChatOllama
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel
from langgraph.graph import StateGraph, START, END
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_ollama import OllamaEmbeddings
from typing import Union
from pydantic import Field

C:\Users\arghy\AppData\Local\Temp\ipykernel_8288\1159099786.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
path = r"C:\Users\arghy\OneDrive\Desktop\3rd sem\1690629458.pdf"
docs = (PyPDFLoader(f'{path}').load())
# print(docs)
chunks = RecursiveCharacterTextSplitter(chunk_size = 900,chunk_overlap = 150).split_documents(docs)
for d in chunks:
    d.page_content = d.page_content.encode("utf-8", "ignore").decode("utf-8", "ignore")
embeddings = OllamaEmbeddings(model='nomic-embed-text')
vector_store = FAISS.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k':4})

result = retriever.invoke('syllabus of 3rd semister')
print("*"*50)
print(result)

**************************************************
[Document(id='b8e6a2ce-60c9-426f-aa85-45448366e5bc', metadata={'producer': 'iLovePDF', 'creator': 'Adobe Acrobat Pro DC 15.23.20070', 'creationdate': '2022-07-06T19:10:01+05:30', 'author': 'Jishan', 'title': '', 'moddate': '2023-07-29T11:15:58+00:00', 'source': 'C:\\Users\\arghy\\OneDrive\\Desktop\\3rd sem\\1690629458.pdf', 'total_pages': 40, 'page': 2, 'page_label': '3'}, page_content='Communication  \n3-0-0-3  3  Proj  PR-IT881  Project-III  0-0-12-12  6  \nOEC  OE-IT702  Open Elective-III \na.Operations Research  \nb. Mobile Computing  \nc. Robotics  \nd.Microwave  \n3-0-0-3  3  Proj  PR-IT882  Viva  0-0-0-0  2  \nProj  PR-IT781  Project-II  0-0-12-12  6  Proj  PR-IT883  Internship Evaluation  0-0-0-0  0  \n Total Credit  14-1-12-27  21     9-0-12-21  17  \n       \n  \n  \n  \n \n \n \n \n \n \n \nJGEC/SYLLABUS/B.TECH./IT/2021-2022               Page 3 of 3'), Document(id='8601e6d3-bf55-4be9-936a-7ede51915424', metadata={'producer'

In [3]:
for i in range(len(result)):
    print(result[i].page_content)

Communication  
3-0-0-3  3  Proj  PR-IT881  Project-III  0-0-12-12  6  
OEC  OE-IT702  Open Elective-III 
a.Operations Research  
b. Mobile Computing  
c. Robotics  
d.Microwave  
3-0-0-3  3  Proj  PR-IT882  Viva  0-0-0-0  2  
Proj  PR-IT781  Project-II  0-0-12-12  6  Proj  PR-IT883  Internship Evaluation  0-0-0-0  0  
 Total Credit  14-1-12-27  21     9-0-12-21  17  
       
  
  
  
 
 
 
 
 
 
 
JGEC/SYLLABUS/B.TECH./IT/2021-2022               Page 3 of 3
JGEC/ 
SYLLABUS 
/ 
B 
. 
TECH 
. 
/IT/ 
20 
21 
- 
20 
                Page  
22 
 1 
  
of  
3 
  
  
  
JALPAIGURI 
- 
  
735102 
  
( 
 ) 
 An Autonomous Government College 
   
  
COURSE   STRUCTURE  AND  SYLLABUS 
  
FOR 
  
FIRST 
  
  
SEMESTER   
T 
O 
  
  
E 
I 
G 
H 
T 
H 
  
  
SEMESTER 
  
  
B.TECH.  
  
D 
E 
G 
R 
E 
E 
  
  
IN  
  
I 
N 
F 
O 
R 
M 
A 
T 
I 
O 
N 
  
  
T 
E 
C 
H 
N 
O 
L 
O 
G 
Y 
  
  
  
( 
Implemented  
for t 
he new  
e 
n 
t 
r 
y 
  
batch  
from the Academic Year 2021 
- 
22 
) 
  
  
  

In [4]:
print(type(result[i].page_content))

<class 'str'>


In [5]:
import re
def decompose_to_sentences(text):
    text = re.sub(r"\s+", " ", text).strip()
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if len(s.strip()) > 20]

In [6]:
strip_list = []
count = 0
for i in range(len(result)):
    z=(decompose_to_sentences(result[i].page_content))
    for j in range(len(z)):
        strip_list.append(z[j])

In [7]:
query = 'syllabus of 3rd semister'
class KeepOrDrop(BaseModel):
    keepordrop:bool = Field(description='return True if the strip is important to answer as per query and False if the sentence is not that important for the query')
llm = ChatOllama(model='qwen2.5:7b',temperature=0)
llm_keepordrop = llm.with_structured_output(KeepOrDrop)
dropping_list = []
for i in range(len(strip_list)):
    z = llm_keepordrop.invoke(f'query:{query},sentence:{strip_list[i]}')
    if not z.keepordrop:
        dropping_list.append(i)

In [10]:
refined_text = ''
for i in range(len(strip_list)):
    if i not in dropping_list:
        refined_text = refined_text + '\n' + strip_list[i]

In [12]:
print(refined_text)


Robotics d.Microwave 3-0-0-3 3 Proj PR-IT882 Viva 0-0-0-0 2 Proj PR-IT781 Project-II 0-0-12-12 6 Proj PR-IT883 Internship Evaluation 0-0-0-0 0 Total Credit 14-1-12-27 21 9-0-12-21 17 JGEC/SYLLABUS/B.TECH./IT/2021-2022 Page 3 of 3
3 Solid Waste: Municipal, industrial, commercial, agricultural, domestic, pathological and hazardous solid wastes; Recovery and disposal method- Open dumping, Land filling, incineration, composting, recycling.
Practical Syllabus Programming in R 1.
Introduction to mechanism for statistics, data analysis, and machine learning; Introduction of R Programming, How to install and run R, Use of R help files, R Sessions, R Objects – Vectors, Attributes, Matrices, Array, Class, List, Data Frames etc.


In [13]:
result = llm.invoke(f'query:{query} and retrieved text:{refined_text}')

In [14]:
print(result.content)

Based on the retrieved text, it appears that the syllabus for the third semester of the Bachelor of Technology in Information Technology (B.Tech. IT) program at JGEC (likely an abbreviation for the college or institution) includes the following courses and practicals:

1. **Robotics (PR-IT882)**
   - Credit: 3
   - Lecture Hours: 3
   - Practical Hours: 0
   - Total Credit: 3

2. **Microwave (PR-IT781)**
   - Credit: 2
   - Lecture Hours: 0
   - Practical Hours: 0
   - Total Credit: 2

3. **Project-II (PR-IT883)**
   - Credit: 6
   - Lecture Hours: 0
   - Practical Hours: 12
   - Total Credit: 6

4. **Internship Evaluation**
   - Credit: 0
   - Lecture Hours: 0
   - Practical Hours: 0
   - Total Credit: 0

5. **Solid Waste Management**
   - This seems to be a theoretical topic rather than a course, covering the following:
     - Types of solid waste: Municipal, industrial, commercial, agricultural, domestic, pathological, and hazardous.
     - Recovery and disposal methods: Open dumpin